Configuration & Security
Objective: Securely initialize the AI environment and connect to the Hugging Face API.
Security: Uses python-dotenv to load the HF_TOKEN from a hidden .env file, ensuring API keys are not hardcoded.
Model: Utilizing Qwen2.5-7B-Instruct via the InferenceClient for high-accuracy health query processing.
UI Libraries: Imports ipywidgets and IPython components to build the interactive chat interface.

In [1]:
import os
import ipywidgets as widgets  # Library for creating buttons and text boxes
from dotenv import load_dotenv  # For loading environment variables from a .env file
from huggingface_hub import InferenceClient  # The client to interact with Hugging Face models
from IPython.display import display, Javascript, HTML  # Tools to render UI and run JS in Notebook

# --- Load Environment Variables ---
# This looks for a .env file in the directory to securely load the API token
load_dotenv()

# --- Access Secret Keys ---
# Using os.getenv prevents the API key from being visible in the code
HF_TOKEN = os.getenv("HF_TOKEN")

# Specifying the LLM to be used for inference
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# --- Initialize Inference Client ---
# Setting up the connection to the Hugging Face API
client = InferenceClient(MODEL_ID, token=HF_TOKEN)

# Confirmation message to ensure the setup worked
print(f"✅ Environment ready: {MODEL_ID} is active.")

✅ Environment ready: Qwen/Qwen2.5-7B-Instruct is active.


Core Logic & Safety
Safety Filter: Blocks high-risk keywords (e.g., self-harm) before AI processing.
Persona: "Senior Medical Assistant" providing professional, structured, and emoji-rich advice.
Response Structure:
Answer: Direct information.
Safety: Actionable precautions.
Summary: Key takeaways.
Disclaimer: Mandatory medical legal notice

In [2]:
def health_chatbot(user_input):
    # Safety Filter (Hardcoded Guardrails) ---
    # Define dangerous keywords to prevent the model from providing harmful advice
    harmful_keywords = ["suicide", "overdose", "lethal dose", "kill myself"]
    
    # Check if the user input contains any prohibited harmful words
    if any(word in user_input.lower() for word in harmful_keywords):
        return "⚠️ I cannot provide information on this. Please contact emergency services immediately."

    # Prompt Engineering (System Persona) ---
    # Provide context to the model regarding its behavior and output format
    messages = [
        {
            "role": "system", 
            "content": (
                "You are a Senior Medical Assistant. Use relevant emojis (like 💊, 👨‍⚕️, ⚠️, ✅) "
                "to make the answer easy to read. Provide detailed, structured, and complete "
                "health information. Format with clear paragraphs and bullet points."
                "\n\nStructure:\n1. Direct Answer\n2. Safety Guidelines\n3. Conclusion\n"
                "4. Disclaimer: 'I am an AI, not a doctor. Consult a professional for diagnosis.'"
            )
        },
        {"role": "user", "content": user_input}
    ]

    # API Call & Error Handling ---
    try:
        # Request response generation (temperature 0.1 ensures higher factual accuracy)
        response = client.chat_completion(messages, max_tokens=1000, temperature=0.1)
        return response.choices[0].message.content
    except Exception as e:
        # Return a formatted error message if the API call fails
        return f"❌ Error: {e}"

Frontend: Interactive UI & UX
Components: Built using ipywidgets for a real-time, responsive chat experience within the Notebook.
Dynamic Styling: Uses HTML/CSS for a clean messaging interface with distinct user and assistant speech bubbles.
UX Features:
Auto-Scroll: Integrated JavaScript to automatically scroll to the latest message.
Live Feedback: Button state changes (loading spinner) while the AI processes the query.
Persistence: chat_history stores the session conversation locally.

In [4]:
#  UI Component Definition ---
chat_history = []  # Internal list to store session messages

# User input text box
text_input = widgets.Text(
    placeholder='Type here and press Enter...', 
    description='💬 Query:', 
    layout={'width': '85%'}
)

# Interaction button with a paper-plane icon
button = widgets.Button(
    description="Send", 
    button_style='success', 
    icon='paper-plane',
    layout={'width': '12%'}
)

# Main chat display area with custom CSS for borders and scrolling
output_html = widgets.HTML(value="<i>Your conversation will appear here...</i>")
output_container = widgets.VBox(
    [output_html], 
    layout={
        'border': '3px solid #28a745', 
        'padding': '15px', 
        'height': '400px', 
        'overflow_y': 'scroll', 
        'width': '100%',
        'background-color': '#ffffff'
    }
)

# Display Rendering Logic ---
def update_display():
    """Generates HTML strings from chat history and triggers auto-scroll."""
    full_chat = ""
    for index, chat in enumerate(chat_history):
        # Formatting user and bot messages with specific colors and styles
        full_chat += f"<div id='msg-{index}' style='margin-bottom: 20px; font-family: sans-serif;'>"
        full_chat += f"<b style='color: #007bff;'>👤 You:</b> {chat['user']}<br>"
        full_chat += f"<div style='margin-top: 5px; background-color: #e9f7ef; padding: 10px; border-radius: 10px;'>"
        full_chat += f"<b style='color: #28a745;'>👨‍⚕️ Assistant:</b><br>{chat['bot'].replace('\n', '<br>')}"
        full_chat += "</div></div>"
    
    output_html.value = full_chat
    
    # Inject JavaScript to handle smooth scrolling to the newest message block
    from IPython.display import display, Javascript
    last_index = len(chat_history) - 1
    if last_index >= 0:
        display(Javascript(f"""
            setTimeout(function() {{
                var target = document.getElementById('msg-{last_index}');
                if (target) {{
                    target.scrollIntoView({{ behavior: 'smooth', block: 'start' }});
                }}
            }}, 150);
        """))

# Query Processing Logic ---
def process_query(sender):
    """Handles the transition between user input and AI response."""
    user_query = text_input.value
    if user_query.strip():
        # Set button to 'Loading' state
        button.description, button.icon, button.button_style = "...", "spinner", 'info'
        
        # Call the core chatbot function (defined in previous step)
        result = health_chatbot(user_query)
        
        # Record the interaction and refresh the UI
        chat_history.append({"user": user_query, "bot": result})
        update_display()
        
        # Reset input and button state
        text_input.value = ""
        button.description, button.icon, button.button_style = "Send", "paper-plane", 'success'

# Event Binding & Layout Display ---
# Trigger processing on click or by pressing the Enter key
button.on_click(process_query)
text_input.on_submit(process_query)

# Render the final layout
display(widgets.VBox([
    widgets.HBox([text_input, button]), 
    output_container
]))

C:\Users\qasim\AppData\Local\Temp\ipykernel_11212\2426420185.py:82: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(process_query)


<IPython.core.display.Javascript object>

Conclusion
Project Summary: Successfully developed a functional AI Health Assistant using the Qwen2.5-7B model via Hugging Face.
Key Achievement: Integrated a robust safety layer to handle sensitive queries and built a responsive UI for a seamless user experience.
Future Scope: Potential improvements include adding multi-language support and integrating a medical knowledge graph for even higher factual accuracy.
Final Note: This project demonstrates the practical application of Prompt Engineering and API Integration in creating safe, specialized AI tools.